# DPS Finetuning Only (No CM)

This notebook is dedicated to tuning DPS hyperparameters only.

Outputs:
- `dps_validation_sweep.csv`: all validation runs
- `dps_best_params.csv`: best `conditioning_scale` per `(noise_sigma, mask_name, steps)`
- `dps_best_params.json`: same best parameters in JSON

You can apply `dps_best_params` to your original DPS-vs-CM comparison notebook.

In [ ]:
import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

%cd /content/drive/MyDrive/Projet_Generative_Models


!ls dps_benchmark.py

import dps_benchmark

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/1urLjhwkcBGyt1y2_PwMIaLPDzLpkfRll/Projet_Generative_Models
dps_benchmark.py


In [ ]:
from dps_benchmark import (
    DPSBedroomRunner,
    build_random_dropout_mask,
    combine_comparison_metrics,
    discuss_method_gap,
    plot_comparison_curve as plot_method_curve,
    run_cm_inpainting_benchmark as run_cm_benchmark_impl,
    run_dps_inpainting_benchmark as run_dps_benchmark_impl,
    save_metrics_table,
    show_dps_grid as show_dps_grid_impl,
    show_method_comparison_grid as show_method_comparison_grid_impl,
    summarize_comparison_table,
)

In [ ]:
!pip install -q lmdb lpips scikit-image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.9/325.9 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 6.3 MB/s eta 0:00:00


In [ ]:
import io
import json
import random
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import lmdb
import numpy as np
import pandas as pd
import torch
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

try:
    import lpips
    _LPIPS_AVAILABLE = True
except ImportError:
    lpips = None
    _LPIPS_AVAILABLE = False

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "dps_finetuning_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR: {BASE_DIR}")
print(f"DEVICE: {DEVICE}")

BASE_DIR: /content/drive/.shortcut-targets-by-id/1urLjhwkcBGyt1y2_PwMIaLPDzLpkfRll/Projet_Generative_Models
DEVICE: cuda


In [ ]:
LSUN_BEDROOM_VAL_URL = "http://dl.yf.io/lsun/scenes/bedroom_val_lmdb.zip"
LSUN_ROOT = BASE_DIR / "lsun_data"
LSUN_VAL_DIR = LSUN_ROOT / "bedroom_val_lmdb"

def ensure_lsun_val_lmdb(root: Path) -> Path:
    root.mkdir(parents=True, exist_ok=True)
    if LSUN_VAL_DIR.exists() and any(LSUN_VAL_DIR.iterdir()):
        return LSUN_VAL_DIR

    zip_path = root / "bedroom_val_lmdb.zip"
    if not zip_path.exists():
        print(f"Downloading LSUN bedroom validation split from {LSUN_BEDROOM_VAL_URL}")
        urlretrieve(LSUN_BEDROOM_VAL_URL, zip_path)

    print(f"Extracting {zip_path} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(root)
    return LSUN_VAL_DIR

def _decode_lsun_value(raw_value: bytes) -> Image.Image:
    return Image.open(io.BytesIO(raw_value)).convert("RGB")

def _pil_to_tensor_minus1_1(img: Image.Image, size: int = 256) -> torch.Tensor:
    img = img.resize((size, size), resample=Image.BICUBIC)
    arr = np.asarray(img).astype(np.float32) / 255.0
    ten = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    return ten * 2.0 - 1.0

def load_references_from_lsun(indices: list[int], size: int = 256):
    lsun_dir = ensure_lsun_val_lmdb(LSUN_ROOT)
    env = lmdb.open(str(lsun_dir), readonly=True, lock=False, readahead=False, max_readers=1)
    refs = []
    with env.begin(write=False) as txn:
        cursor = txn.cursor()
        keys = [k for k, _ in cursor]
        total = len(keys)
        for idx in indices:
            if idx < 0 or idx >= total:
                raise IndexError(f"Index {idx} out of range (0..{total-1})")
            raw = txn.get(keys[idx])
            pil = _decode_lsun_value(raw)
            tensor = _pil_to_tensor_minus1_1(pil, size=size)
            refs.append({"index": idx, "source": "lsun_val", "tensor": tensor})
    env.close()
    return refs

def make_center_box_mask(size: int = 256, box_ratio: float = 0.5) -> torch.Tensor:
    mask = torch.zeros((1, 1, size, size), dtype=torch.float32)
    box = int(size * box_ratio)
    s = (size - box) // 2
    e = s + box
    mask[:, :, s:e, s:e] = 1.0
    return mask

def make_noisy_observation(reference: torch.Tensor, missing_mask: torch.Tensor, noise_sigma: float, seed: int) -> torch.Tensor:
    device = reference.device
    known_mask = (1.0 - missing_mask).to(device)
    obs = reference * known_mask
    if noise_sigma > 0:
        g = torch.Generator(device=device).manual_seed(seed)
        noise = torch.randn(reference.shape, generator=g, device=device) * noise_sigma
        obs = (reference + noise) * known_mask
    return obs.clamp(-1.0, 1.0)

def tensor_to_numpy_image(t: torch.Tensor) -> np.ndarray:
    if t.ndim == 4:
        t = t[0]
    t = t.detach().cpu().clamp(-1.0, 1.0)
    t = (t + 1.0) / 2.0
    return t.permute(1, 2, 0).numpy()

In [ ]:
VALIDATION_REFERENCE_INDICES = [0, 48]
NOISE_LEVELS = [0.0, 0.02, 0.05]
DPS_STEPS = [100]
DPS_CONDITIONING_SCALES = [0.1, 0.25, 0.5, 1.0, 2.0]
IMAGE_SIZE = 256

comparison_masks = {
    "center_box": make_center_box_mask(size=IMAGE_SIZE, box_ratio=0.5),
}

validation_entries = load_references_from_lsun(VALIDATION_REFERENCE_INDICES, size=IMAGE_SIZE)
print(f"Validation images: {[e['index'] for e in validation_entries]}")
print(f"Masks: {list(comparison_masks.keys())}")
print(f"Noise levels: {NOISE_LEVELS}")
print(f"DPS steps: {DPS_STEPS}")
print(f"Scale grid: {DPS_CONDITIONING_SCALES}")

Validation images: [0, 48]
Masks: ['center_box']
Noise levels: [0.0, 0.02, 0.05]
DPS steps: [100]
Scale grid: [0.1, 0.25, 0.5, 1.0, 2.0]


In [ ]:
from dps_benchmark import DPSBedroomRunner

# Manually define repo paths if needed or let runner handle it
dps_repo_path = BASE_DIR / "diffusion-posterior-sampling"

# If git clone fails because dir exists, we can try to proceed or clean up
if not dps_repo_path.exists():
    print(f"Cloning DPS repo to {dps_repo_path}...")
    !git clone https://github.com/DPS2022/diffusion-posterior-sampling.git {dps_repo_path}
else:
    print(f"DPS repo already exists at {dps_repo_path}")

runner = DPSBedroomRunner(
    base_dir=BASE_DIR,
    device=DEVICE,
    in_colab=True,
    use_fp16=True,
    seed=SEED,
)

runner.load_prior(force_reload=False)

lpips_metric = None
if _LPIPS_AVAILABLE:
    import lpips
    lpips_metric = lpips.LPIPS(net="alex").to(DEVICE)

print("DPS prior is ready.")

DPS repo already exists at /content/drive/.shortcut-targets-by-id/1urLjhwkcBGyt1y2_PwMIaLPDzLpkfRll/Projet_Generative_Models/diffusion-posterior-sampling
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
DPS prior is ready.


In [ ]:
def compute_dps_metrics(reference: torch.Tensor, reconstructed: torch.Tensor, missing_mask: torch.Tensor, lpips_model=None):
    # Inputs are in [-1, 1], shape [1, 3, H, W].
    ref = reference.detach().float().cpu()
    rec = reconstructed.detach().float().cpu()
    miss = missing_mask.detach().float().cpu()
    known = 1.0 - miss

    ref_np = tensor_to_numpy_image(ref)
    rec_np = tensor_to_numpy_image(rec)

    miss_3 = miss.repeat(1, 3, 1, 1)
    known_3 = known.repeat(1, 3, 1, 1)

    miss_pixels = miss_3.sum().item()
    known_pixels = known_3.sum().item()

    missing_l1 = (torch.abs(ref - rec) * miss_3).sum().item() / max(miss_pixels, 1.0)
    known_l1 = (torch.abs(ref - rec) * known_3).sum().item() / max(known_pixels, 1.0)

    miss_np = miss[0, 0].numpy() > 0.5
    ref_miss_np = ref_np[miss_np]
    rec_miss_np = rec_np[miss_np]

    if ref_miss_np.size > 0:
        dr = 1.0
        mse = np.mean((ref_miss_np - rec_miss_np) ** 2)
        missing_psnr = peak_signal_noise_ratio(ref_miss_np, rec_miss_np, data_range=dr) if mse > 0 else float("inf")
    else:
        missing_psnr = np.nan

    # Full-image SSIM as stable fallback.
    missing_ssim = structural_similarity(ref_np, rec_np, channel_axis=2, data_range=1.0)

    missing_lpips = np.nan
    if lpips_model is not None:
        with torch.no_grad():
            r = ref.to(DEVICE)
            x = rec.to(DEVICE)
            missing_lpips = float(lpips_model(r, x).mean().item())

    return {
        "missing_region_l1": float(missing_l1),
        "known_region_l1": float(known_l1),
        "missing_region_psnr": float(missing_psnr) if np.isfinite(missing_psnr) else np.nan,
        "full_image_ssim": float(missing_ssim),
        "missing_region_lpips": float(missing_lpips) if np.isfinite(missing_lpips) else np.nan,
    }

def tuning_objective_col(df: pd.DataFrame) -> str:
    return "missing_region_lpips" if "missing_region_lpips" in df.columns and df["missing_region_lpips"].notna().any() else "missing_region_l1"

In [ ]:
def run_dps_validation_sweep(
    runner: DPSBedroomRunner,
    validation_entries: list[dict],
    comparison_masks: dict[str, torch.Tensor],
    noise_levels: list[float],
    steps: list[int],
    scale_grid: list[float],
    seed: int,
    lpips_model=None,
) -> pd.DataFrame:
    rows = []
    for scale in scale_grid:
        print(f"Running scale={scale} ...")
        for ref in validation_entries:
            reference = ref["tensor"]
            for mask_name, missing_mask in comparison_masks.items():
                for noise_sigma in noise_levels:
                    for step in steps:
                        edited, _, wall_clock_s = runner.run(
                            image=reference,
                            missing_mask=missing_mask,
                            num_inference_steps=step,
                            noise_sigma=float(noise_sigma),
                            conditioning_scale=float(scale),
                            seed=int(seed),
                            make_noisy_observation=make_noisy_observation,
                            record_history=False,
                        )

                        metrics = compute_dps_metrics(reference, edited, missing_mask, lpips_model=lpips_model)
                        metrics.update({
                            "reference_index": int(ref["index"]),
                            "mask_name": str(mask_name),
                            "noise_sigma": float(noise_sigma),
                            "steps": int(step),
                            "conditioning_scale": float(scale),
                            "wall_clock_s": float(wall_clock_s),
                        })
                        rows.append(metrics)

    df = pd.DataFrame(rows).sort_values(["reference_index", "mask_name", "noise_sigma", "steps", "conditioning_scale"]).reset_index(drop=True)
    return df

dps_validation_sweep_df = run_dps_validation_sweep(
    runner=runner,
    validation_entries=validation_entries,
    comparison_masks=comparison_masks,
    noise_levels=NOISE_LEVELS,
    steps=DPS_STEPS,
    scale_grid=DPS_CONDITIONING_SCALES,
    seed=SEED,
    lpips_model=lpips_metric,
)

objective_col = tuning_objective_col(dps_validation_sweep_df)
print(f"Tuning objective: {objective_col}")

group_cols = ["noise_sigma", "mask_name", "steps", "conditioning_scale"]
summary_df = dps_validation_sweep_df.groupby(group_cols, dropna=False)[[objective_col, "wall_clock_s"]].mean().reset_index()
best_idx = summary_df.groupby(["noise_sigma", "mask_name", "steps"], dropna=False)[objective_col].idxmin()
dps_best_params_df = summary_df.loc[best_idx].sort_values(["noise_sigma", "mask_name", "steps"]).reset_index(drop=True)

display(dps_best_params_df)

Running scale=0.1 ...
Running scale=0.25 ...
Running scale=0.5 ...
Running scale=1.0 ...
Running scale=2.0 ...
Tuning objective: missing_region_lpips


,noise_sigma,mask_name,steps,conditioning_scale,missing_region_lpips,wall_clock_s
0,0.00,center_box,100,2.0,0.743123,61.643689
1,0.02,center_box,100,2.0,0.743863,61.648225
2,0.05,center_box,100,2.0,0.749919,61.688943


In [ ]:
sweep_csv = OUTPUT_DIR / "dps_validation_sweep.csv"
best_csv = OUTPUT_DIR / "dps_best_params.csv"
best_json = OUTPUT_DIR / "dps_best_params.json"

dps_validation_sweep_df.to_csv(sweep_csv, index=False)
dps_best_params_df.to_csv(best_csv, index=False)

best_lookup = {
    f"sigma={row.noise_sigma:.3f}|mask={row.mask_name}|steps={int(row.steps)}": float(row.conditioning_scale)
    for row in dps_best_params_df.itertuples(index=False)
}
with open(best_json, "w", encoding="utf-8") as f:
    json.dump(best_lookup, f, indent=2)

print(f"Saved sweep table: {sweep_csv}")
print(f"Saved best params CSV: {best_csv}")
print(f"Saved best params JSON: {best_json}")
print(best_lookup)

Saved sweep table: /content/drive/.shortcut-targets-by-id/1urLjhwkcBGyt1y2_PwMIaLPDzLpkfRll/Projet_Generative_Models/dps_finetuning_outputs/dps_validation_sweep.csv
Saved best params CSV: /content/drive/.shortcut-targets-by-id/1urLjhwkcBGyt1y2_PwMIaLPDzLpkfRll/Projet_Generative_Models/dps_finetuning_outputs/dps_best_params.csv
Saved best params JSON: /content/drive/.shortcut-targets-by-id/1urLjhwkcBGyt1y2_PwMIaLPDzLpkfRll/Projet_Generative_Models/dps_finetuning_outputs/dps_best_params.json
{'sigma=0.000|mask=center_box|steps=100': 2.0, 'sigma=0.020|mask=center_box|steps=100': 2.0, 'sigma=0.050|mask=center_box|steps=100': 2.0}
